# Python Code Generator + Test Case Generator
## Built with LangGraph + Groq

```
User Request
      ↓
Requirements Extractor
      ↓
Code Generator Agent  ←──────────────────┐
      ↓                                  │  retry (max 3)
Test Generator Agent                     │
      ↓                                  │
Code Executor (uses RunCode Tool)  ──[Fail]──┘
      ↓
   [Pass]
      ↓
   Response
```

## Cell 1: Imports

In [ ]:
import os
import re
import sys
import subprocess
import tempfile
import textwrap
import getpass
from typing import TypedDict, Optional, Dict

from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END

import gradio as gr

print('All imports successful')

## Cell 2: API Key & LLM Setup

In [ ]:
os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
print('LLM ready')

## Cell 3: SimpleAgent Class

Same pattern as the instructor's reference notebook. Each agent has an LLM, a list of tools, and a system prompt.
The agent:
1. Asks the LLM which tool to use
2. Extracts the right parameter for that tool
3. Calls the tool
4. Asks the LLM to generate a final response using the tool result

In [ ]:
class SimpleAgent:
    """
    Simplified agent: LLM + tools + system prompt.
    Mirrors the instructor's SimpleAgent pattern.
    """
    def __init__(self, llm, tools: list, system_prompt: str, agent_name: str = 'Agent'):
        self.llm          = llm
        self.tools        = {tool.name: tool for tool in tools}
        self.system_prompt = system_prompt
        self.agent_name   = agent_name

    def invoke(self, inputs: Dict[str, str]) -> Dict[str, str]:
        query = inputs.get('input', '')

        # Step 1 — ask LLM which tool to use
        prompt = f"""{self.system_prompt}

Available Tools:
{self._format_tools()}

User Request: {query}

Think step by step:
- Which tool do you need (if any)?
- What exact input should be passed to it?"""

        response = self.llm.invoke(prompt).content

        # Step 2 — if a tool is mentioned, extract param and call it
        for tool_name, tool in self.tools.items():
            if tool_name.lower() in response.lower():
                extraction_prompt = f"""From this request:
\"{query}\"

Extract ONLY the exact input to pass to the {tool_name} tool.
Return just the value, nothing else."""
                param = self.llm.invoke(extraction_prompt).content.strip().strip('"').strip("'")
                tool_result = tool.func(param)

                # Step 3 — final answer using tool result
                final_prompt = f"""{self.system_prompt}

User Request: {query}
Tool Used: {tool_name}
Tool Result:
{tool_result}

Now provide your final response based on the tool result:"""
                return {'output': self.llm.invoke(final_prompt).content}

        # No tool needed — return direct LLM response
        return {'output': response}

    def _format_tools(self) -> str:
        if not self.tools:
            return 'No tools available — respond directly.'
        return '\n'.join(f'- {t.name}: {t.description}' for t in self.tools.values())


print('SimpleAgent class ready')

## Cell 4: Tools

`run_code_tool` — executes Python code in a subprocess sandbox and returns stdout/stderr.
This is a **Tool** (not an agent), used by the Code Executor node.

In [ ]:
def run_python_code(code: str) -> str:
    """
    Executes Python code in a subprocess sandbox.
    Returns combined stdout + stderr.
    """
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(code)
        tmp_path = f.name
    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True, text=True, timeout=15
        )
        return (result.stdout + result.stderr).strip(), result.returncode
    except subprocess.TimeoutExpired:
        return 'Error: Code execution timed out (15s limit)', 1


# Wrap as a LangChain Tool
run_code_tool = Tool(
    name='RunCode',
    func=lambda code: run_python_code(code)[0],   # tool only needs the string output
    description='Executes Python code in a sandbox and returns stdout/stderr. '
                'Input must be complete, runnable Python code.'
)

print('RunCode tool ready')

## Cell 5: Agents

Two **SimpleAgent** instances — one for code generation, one for test generation.

In [ ]:
# ── Agent 1: Code Generator ───────────────────────────────────────────────────
code_generator_agent = SimpleAgent(
    llm=llm,
    tools=[],          # pure LLM generation, no tools needed
    system_prompt="""You are an expert Python developer. Your ONLY job is to write correct, clean Python code.

When given requirements:
1. Read the requirements carefully, including any edge cases
2. Write clean, Pythonic code with no placeholder comments
3. Handle all edge cases mentioned
4. Return ONLY a ```python ... ``` code block — no explanation

If you receive a previous error, analyse it and fix the root cause.""",
    agent_name='Code Generator Agent'
)

# ── Agent 2: Test Generator ───────────────────────────────────────────────────
test_generator_agent = SimpleAgent(
    llm=llm,
    tools=[],          # pure LLM generation
    system_prompt="""You are a Python testing expert. Your ONLY job is to write pytest-style test cases.

When given code to test:
1. Write 4-6 test functions, each named test_*
2. Cover: happy path, edge cases, boundary values, error cases
3. Use plain assert statements
4. Do NOT import the code under test — it will be injected above your tests automatically
5. Return ONLY a ```python ... ``` code block — no explanation""",
    agent_name='Test Generator Agent'
)

print('Code Generator Agent ready')
print('Test Generator Agent ready')

## Cell 6: LangGraph State

In [ ]:
class CodeGenState(TypedDict):
    user_request:      str    # original natural-language description
    requirements:      str    # structured spec from requirements extractor
    generated_code:    str    # code produced by code generator agent
    test_code:         str    # tests produced by test generator agent
    execution_result:  str    # stdout/stderr from RunCode tool
    execution_success: bool   # True if all tests passed
    retry_count:       int    # incremented by executor on failure
    error_message:     str    # last error, fed back to code generator agent
    final_response:    str    # formatted answer shown to user

MAX_RETRIES = 3

def extract_code(text: str) -> str:
    """Strip markdown code fences from LLM output."""
    match = re.search(r'```(?:python)?\n(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

print('State schema defined')

## Cell 7: Graph Nodes

Each node is a thin wrapper that calls the right agent/tool and updates state.

| Node | Calls | Role |
|---|---|---|
| `requirements_extractor` | LLM directly | Extracts structured spec |
| `code_generator_node` | `code_generator_agent` | Generates / refines Python code |
| `test_generator_node` | `test_generator_agent` | Generates pytest test cases |
| `code_executor_node` | `run_code_tool` | Runs code + tests, updates retry_count |
| `response_node` | — | Formats final output |

In [ ]:
# ── Node 1: Requirements Extractor ───────────────────────────────────────────
def requirements_extractor_node(state: CodeGenState) -> dict:
    print('[REQUIREMENTS] Extracting structured spec...')
    prompt = f"""You are a software requirements analyst.
Extract a structured specification from the user request.

User request: {state['user_request']}

Return ONLY this format:
TASK_TYPE: <function | class | script>
NAME: <name>
DESCRIPTION: <one sentence>
PARAMETERS: <param: type, ... or None>
RETURNS: <return type and description>
EDGE_CASES: <comma-separated edge cases>"""

    requirements = llm.invoke(prompt).content.strip()
    print(f'[REQUIREMENTS]\n{requirements}\n')
    return {'requirements': requirements}


# ── Node 2: Code Generator (uses Code Generator Agent) ───────────────────────
def code_generator_node(state: CodeGenState) -> dict:
    print(f'[CODE GENERATOR AGENT] Generating code (retry={state.get("retry_count", 0)})...')

    agent_input = f'Requirements:\n{state["requirements"]}'

    # On retry: pass the error back so the agent can fix the code
    if state.get('error_message') and state.get('retry_count', 0) > 0:
        agent_input += (
            f'\n\nPrevious attempt FAILED. Fix these errors:\n{state["error_message"]}'
            f'\n\nFailed code:\n```python\n{state["generated_code"]}\n```'
        )

    result = code_generator_agent.invoke({'input': agent_input})
    code   = extract_code(result['output'])
    print(f'[CODE GENERATOR AGENT] Done — {len(code.splitlines())} lines.')
    return {'generated_code': code, 'error_message': ''}


# ── Node 3: Test Generator (uses Test Generator Agent) ───────────────────────
def test_generator_node(state: CodeGenState) -> dict:
    print('[TEST GENERATOR AGENT] Generating test cases...')

    agent_input = (
        f'Write pytest tests for this code.\n\n'
        f'Code:\n```python\n{state["generated_code"]}\n```\n\n'
        f'Requirements (for reference):\n{state["requirements"]}'
    )

    result    = test_generator_agent.invoke({'input': agent_input})
    test_code = extract_code(result['output'])
    print(f'[TEST GENERATOR AGENT] Done — {len(test_code.splitlines())} lines.')
    return {'test_code': test_code}


# ── Node 4: Code Executor (uses RunCode Tool) ────────────────────────────────
def code_executor_node(state: CodeGenState) -> dict:
    print('[CODE EXECUTOR] Running code + tests via RunCode tool...')

    # Build combined script: implementation + tests + auto-runner
    combined = textwrap.dedent(f"""
{state['generated_code']}

{state['test_code']}

# ── Auto-run all test_ functions ──
import sys as _sys
_passed = _failed = 0
for _name, _fn in list(globals().items()):
    if _name.startswith('test_') and callable(_fn):
        try:
            _fn()
            print(f'  PASS  {{_name}}')
            _passed += 1
        except Exception as _e:
            print(f'  FAIL  {{_name}}: {{_e}}')
            _failed += 1
print(f'\\nResults: {{_passed}} passed, {{_failed}} failed')
if _failed > 0:
    _sys.exit(1)
    """).strip()

    # Call the RunCode tool
    output, returncode = run_python_code(combined)
    success = returncode == 0

    print(f'[CODE EXECUTOR] Success={success}')
    print(f'[CODE EXECUTOR]\n{output}\n')

    # Increment retry count on failure (replaces the dummy refinement node)
    retry_count = state.get('retry_count', 0)
    if not success:
        retry_count += 1

    return {
        'execution_result':  output,
        'execution_success': success,
        'error_message':     '' if success else output,
        'retry_count':       retry_count,
    }


# ── Node 5: Response Formatter ────────────────────────────────────────────────
def response_node(state: CodeGenState) -> dict:
    print('[RESPONSE] Formatting final output...')
    status = 'All tests passed' if state['execution_success'] else 'Tests did not pass'
    retries = state.get('retry_count', 0)

    response = (
        f'## Generated Code\n'
        f'```python\n{state["generated_code"]}\n```\n\n'
        f'## Test Cases\n'
        f'```python\n{state["test_code"]}\n```\n\n'
        f'## Execution Result\n'
        f'```\n{state["execution_result"]}\n```\n\n'
        f'**{status}** after {retries} retry/retries.'
    )
    return {'final_response': response}


print('All nodes defined')

## Cell 8: Routing (Conditional Edge)

After the executor runs:
- **Pass** → `response_node`  
- **Fail + retries left** → back to `code_generator_node` (no dummy refinement node needed)  
- **Fail + max retries** → `response_node` with error

In [ ]:
def route_after_executor(state: CodeGenState) -> str:
    if state['execution_success']:
        print('[ROUTER] Tests PASSED → response')
        return 'response_node'
    if state.get('retry_count', 0) < MAX_RETRIES:
        print(f'[ROUTER] Tests FAILED (retry {state["retry_count"]}/{MAX_RETRIES}) → code_generator')
        return 'code_generator_node'
    print('[ROUTER] Max retries reached → response')
    return 'response_node'

print('Routing function ready')

## Cell 9: Build & Compile the LangGraph

In [ ]:
builder = StateGraph(CodeGenState)

# ── Nodes ─────────────────────────────────────────────────────────────────────
builder.add_node('requirements_extractor', requirements_extractor_node)
builder.add_node('code_generator_node',    code_generator_node)
builder.add_node('test_generator_node',    test_generator_node)
builder.add_node('code_executor_node',     code_executor_node)
builder.add_node('response_node',          response_node)

# ── Linear edges ──────────────────────────────────────────────────────────────
builder.set_entry_point('requirements_extractor')
builder.add_edge('requirements_extractor', 'code_generator_node')
builder.add_edge('code_generator_node',    'test_generator_node')
builder.add_edge('test_generator_node',    'code_executor_node')
builder.add_edge('response_node',          END)

# ── Conditional edge: executor → pass or retry loop ───────────────────────────
builder.add_conditional_edges(
    'code_executor_node',
    route_after_executor,
    {
        'response_node':       'response_node',
        'code_generator_node': 'code_generator_node',   # retry loop, no refinement node
    }
)

graph = builder.compile()
print('LangGraph compiled successfully!')

## Cell 10: LangGraph Flow Diagram

In [ ]:
try:
    from IPython.display import display, Image
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## Cell 11: Helper

In [ ]:
def generate_code(user_request: str) -> dict:
    initial_state: CodeGenState = {
        'user_request':      user_request,
        'requirements':      '',
        'generated_code':    '',
        'test_code':         '',
        'execution_result':  '',
        'execution_success': False,
        'retry_count':       0,
        'error_message':     '',
        'final_response':    '',
    }
    return graph.invoke(initial_state)

print('generate_code() helper ready')

## Cell 12: Test Examples

In [ ]:
result = generate_code('Write a function that checks if a number is prime')
print(result['final_response'])

In [ ]:
result = generate_code('Write a function that finds the two numbers in a list that add up to a target sum')
print(result['final_response'])

## Cell 13: Gradio Chat UI

In [ ]:
def gradio_generate(message: str, history: list) -> str:
    try:
        result = generate_code(message)
        return result['final_response']
    except Exception as e:
        return f'Error: {str(e)}'


demo = gr.ChatInterface(
    fn=gradio_generate,
    title='Python Code Generator + Test Case Generator',
    description='Describe what you want in plain English. The system generates Python code, test cases, executes them, and auto-refines on failures.',
    examples=[
        'Write a function to check if a string is a palindrome',
        'Write a function that finds the factorial of a number recursively',
        'Write a function that merges two sorted lists into one sorted list',
        'Write a function that counts the frequency of each word in a string',
    ],
)

demo.launch(debug=True, share=True)